***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 3 章：干涉测量中的位置天文学](3_0_introduction.ipynb)
    * 上一节：[3.4 方向余弦、相位中心与局部成像坐标](3_4_direction_cosine_coordinates.ipynb)
    * 下一节：[3.P 定量问题集](3_problem_set.ipynb)

***


## 3.5 时间标准、参考系与精密天体测量边界

前几节已经把赤道坐标、时角、地平坐标和方向余弦坐标连接起来。对普通成像而言，这条几何链通常可以被当作“软件已经处理好的坐标转换”；但在高频相位参考、长基线干涉、宽场拼接和精密天体测量中，时间标准、地球参考系和源目录参考架会直接进入相位。几何误差一旦被乘上观测频率，就不再只是一个很小的坐标误差，而会变成可见度相位、延迟和像面位置的系统偏差。

干涉测量中的相位误差可以从几何延迟看出。若基线向量为 $\mathbf{b}$，天空方向为 $\mathbf{s}$，几何延迟近似为

$$
\tau_g = {\mathbf{b}\cdot\mathbf{s}\over c}.
$$

当延迟模型存在小误差 $\Delta\tau$ 时，频率 $\nu$ 上的相位误差幅度为

$$
|\Delta\phi| = 2\pi\nu|\Delta\tau|.
$$

相位的正负号取决于相关器的指数约定和基线顺序，本节只讨论误差幅度。这个式子说明时间和几何为什么在高频与长基线观测中变得尖锐。皮秒量级的延迟误差在厘米波至毫米波相位中已经可见；纳秒量级误差在 GHz 频段会跨越多个相位周。精密天体测量的本质，是把这类延迟和相位误差压到能够支持目标角位置测量的水平。

![时间标准、参考系与几何模型链条](figures/time_reference_chain.png)

**图 3.5.1** 时间标准、参考系和干涉几何之间的链条。`UTC` 适合记录观测时间，`TAI/TT` 适合动力学计算，`UT1` 和地球定向参数决定地球相对于惯性空间的转角。源目录所在的天球参考架、台站所在的地球参考架和观测时刻必须共同进入延迟模型。

### 三个精度层级：公式演示不等于延迟模型

同一个坐标变换可以服务于不同目标，但输入数据、验证方法和允许结论不能混用。下表不给出普适角秒门限，因为相位容差还随频率、基线和动态范围改变；它固定的是每个层级最低限度的证据。

| 层级 | 目标 | 可以采用 | 必须补入 | 可支持的结论 |
|:---|:---|:---|:---|:---|
| 概念级 | 理解 RA/Dec、LST/时角、Alt/Az 和 $(l,m,n)$ 的几何与符号 | 教学历元、近似恒星时、球面三角和 WGS84 米级例子 | 单位、东/西与正负号检查，和一个数量级核对 | 可观测性、坐标链和相位标度；不能据此声称真实数据的绝对相位或天体位置 |
| 常规成像级 | 让相关器元数据、`uvw`、相位中心和 WCS 在同一模型中一致 | Astropy/ERFA/SOFA 等标准实现和阵列软件 | 观测历元、台站坐标、IERS EOP、频率定义、相位旋转记录以及图像位置交叉核对 | 在合成波束和校准误差允许范围内解释形态与相对位置；不能自动升级为亚毫角秒天体测量 |
| 精密天体测量级 | 解释差分延迟、VLBI 相位参考、视差或自行 | 经验证的专业延迟模型与 fringe fitting | 版本化 ITRF/ICRF、站速度和位移、最终/预测 EOP 身份、时钟、对流层/电离层、校准源结构与目标-校准源角距的误差预算 | 只有通过残余 delay/rate、校准源留出和多历元稳定性检验后，才能报告绝对或差分天体位置 |

因此，“调用了高精度库”只是常规成像的必要条件之一，不等于已经完成精密天体测量；反过来，概念级公式仍然适合检查软件输出的符号、数量级和失败边界。

### 3.5.1 时间标准为什么不只是时间戳

观测日志通常以 `UTC` 记录，因为它与民用时间和闰秒制度相连，便于调度和归档；但它不是适合所有计算的单一自变量。`TAI` 是连续原子时标，`TT=TAI+32.184\,\mathrm{s}`，常用于地心星历和动力学参数；`UT1` 跟踪地球实际自转。`UTC` 在插入闰秒时不连续，`TAI-UTC` 随闰秒阶跃，而 `UT1-UTC` 必须从国际地球自转和参考系服务（IERS）发布的地球定向参数取得。

地球自转角 `ERA` 由 `UT1` 儒略日直接定义：

$$
\mathrm{ERA}=2\pi\left[0.7790572732640+1.00273781191135448\left(\mathrm{JD}_{\rm UT1}-2451545.0\right)\right]\pmod{2\pi}.
$$

`GMST/GAST` 在 ERA 基础上加入岁差以及章动相关量，再与向东为正的台站经度相加得到地方恒星时。因此时角

$$
H = \mathrm{LST}-\alpha
$$

中的 `LST` 并不是从 UTC 钟表读数做一个固定比例换算就能得到的。

地球定向参数（EOP）至少包括 `UT1-UTC`、极移 $x_p,y_p$，以及天球中间极相对于模型的改正 $dX,dY$。标准软件会调用 IERS 数据完成转换，但数据版本仍须可追溯；若 EOP 缺失或被长期预测值替代，同一基线会被投影到略有不同的 `uvw` 坐标，随后表现为残余延迟、fringe rate 或像面位置偏差。闰秒附近的时间解析还必须让时间库处理，不能手写成“每天恒有 86400 个 UTC 秒”。

对连通阵列，常见影响首先出现在高频长基线相位稳定性和精确相位中心定义中；对 VLBI，站钟偏移和漂移本身就是待估参数。fringe fitting 中的残余 delay 和 fringe rate 可以被看作对几何、时钟和传播介质模型剩余误差的联合修正，但拟合成功并不等于绝对天体测量参考已经正确。

![延迟误差对相位和天体测量的影响](figures/delay_phase_astrometry_scale.png)

**图 3.5.2** 左图给出不同频率下延迟误差对应的相位误差；右图给出同一延迟误差在不同基线长度上对应的角位置误差量级。频率越高，相位对延迟越敏感；基线越长，同一延迟误差对应的角位置越小，因此长基线可以提供高角分辨率，也要求更严格的几何和时钟模型。

### 3.5.2 天球参考架、地球参考架与相位中心

源目录中的赤经、赤纬通常被理解为天球上的固定方向，但“固定”并不意味着可以脱离参考架。`ICRS` 是理想天球参考系统，`ICRF` 是由河外射电源位置实现的参考架；`ITRS` 是随地球固连的参考系统，`ITRF20xx` 则是由站坐标和速度实现的具体版本。`GCRS/ITRS` 之间的观测时刻变换需要岁差-章动、ERA、极移和 EOP；恒星还要加入自行、视差、光行差和引力偏折，太阳系目标则应由星历给出时变方向。

在普通成像 Notebook 中，常把参考方向写成 $(\alpha_0,\delta_0)$，再用 $(l,m,n)$ 描述偏移。相关器按延迟跟踪中心消除名义几何延迟；若成像参考方向不同，必须执行一致的相位旋转并更新 `uvw/WCS`。一个固定小位置偏移产生随基线变化的可见度相位坡度，并在图像中表现为位置偏移；只有当误差还随时间、频率或天线变化时，才进一步造成展宽、漂移或方向相关残差。

因此，精确位置天文学不只是“坐标公式更复杂”。它关心的是：目录方向、台站坐标、地球姿态、传播介质、仪器时钟和相关器模型是否共同定义了同一个几何问题。只要其中某一环的约定不一致，校准和成像可能仍能给出看似合理的图像，但图像的绝对位置、相对位置或频率相关位置就不再可直接解释。

### 3.5.3 台站坐标：大地坐标、ECEF 与 ENU

台站位置常以大地纬度 $\varphi$、经度 $\lambda$ 和椭球高 $h$ 给出，也可写成地心地固直角坐标 `ECEF`。ECEF 的 $X$ 轴穿过赤道与零经度，$Y$ 轴穿过赤道与 $90^\circ$E，$Z$ 轴指向约定地球北极。必须区分大地纬度与地心纬度：前者是参考椭球法线与赤道面的夹角，后者是地心径向量的夹角；除赤道和两极外二者不同。椭球高也不是相对于平均海平面的正高。

WGS84 椭球可用于本书的米级几何示例，[ecef.py](ecef.py) 给出了双向转换；精密阵列和 VLBI 应使用注明版本与参考历元的 `ITRF` 站坐标、速度和天线参考点，并按需要加入板块运动、固体潮、海潮负载与大气负载等位移。两天线的基线必须先在同一参考架和同一历元中作差。

在台站处，把 ECEF 差向量 $\Delta\mathbf r=(\Delta X,\Delta Y,\Delta Z)^T$ 转到局部东-北-天顶（ENU）坐标，可使用

$$
\begin{pmatrix}E\\N\\U\end{pmatrix}=
\begin{pmatrix}
-\sin\lambda & \cos\lambda & 0\\
-\sin\varphi\cos\lambda & -\sin\varphi\sin\lambda & \cos\varphi\\
\cos\varphi\cos\lambda & \cos\varphi\sin\lambda & \sin\varphi
\end{pmatrix}
\begin{pmatrix}\Delta X\\\Delta Y\\\Delta Z\end{pmatrix}.
$$

这里 $\varphi$ 必须是大地纬度，经度采用向东为正。该矩阵是正交旋转，因此保持向量长度。地心处大地坐标没有定义；在两极，经度以及局部东、北方向依赖所选子午线约定，而天顶方向仍可定义。可靠代码应显式测试这些边界。


### 3.5.4 精密天体测量的误差边界

理想情况下，点源位置的热噪声误差常用合成波束宽度和信噪比估计：

$$
\sigma_\theta \simeq {\theta_{\rm beam}\over 2\,\mathrm{SNR}}.
$$

这个关系只是在孤立、未分辨源和近似高斯波束条件下的统计误差量级，系数还会随波束形状、拟合方法和相关噪声改变。真实相位参考观测还受到校准源位置、目标与校准源角距离、对流层或电离层残余路径、天线位置、频率相关源结构、核心位移、时间插值和方向相关误差的限制。热噪声误差可以通过更高信噪比下降，但许多系统误差不会以同样方式平均掉。

长基线观测尤其容易展示这种差异。更长基线使 $\theta_{\rm beam}\sim\lambda/B$ 变小；同一投影方向上的小相位误差对应

$$
|\Delta\theta|\simeq{\lambda|\Delta\phi|\over2\pi B}={c|\Delta\tau|\over B}.
$$

相位参考的目标不是让每个可见度相位都接近零，而是让目标源和参考源之间的差分相位能够被稳定解释。电离层群延迟近似随 $\nu^{-2}$ 变化、相位随 $\nu^{-1}$ 变化；中性大气路径在远离谱线时近似非色散，但相位随 $\nu$ 增大。若目标和参考源相距过远，两条视线不再共享同一传播误差；若参考源本身有结构或核心位移，绝对位置也会被参考源定义所限制。

#### 数值检查：同一 $10^\circ$ 相位预算对应什么误差

对完全沿投影基线方向的位置误差，$|\Delta\phi|=2\pi(B/\lambda)|\Delta\theta|$。若把 `UT1` 时间误差只看作地球转角误差，则最不利投影给出 $|\Delta\tau|\lesssim B\omega_\oplus|\Delta t|/c$。下面反求三类观测在 $10^\circ$ 相位预算下的位置、几何延迟和 `UT1` 误差上限。`UT1` 一列是保守投影上限，不是调度时间戳或 fringe fitting 解的通用容差。


In [ ]:
import numpy as np

C_LIGHT = 299_792_458.0
OMEGA_EARTH = 7.2921150e-5
phase_limit = np.deg2rad(10.0)
cases = [
    ('cm connected', 36e3, 1.4e9),
    ('mm connected', 16e3, 100e9),
    ('VLBI', 5e6, 8.4e9),
]

print('case            position [mas]  delay [ps]  UT1 bound [s]')
for name, baseline, frequency in cases:
    angle = phase_limit * C_LIGHT / (2 * np.pi * frequency * baseline)
    angle_mas = np.rad2deg(angle) * 3.6e6
    delay_ps = phase_limit / (2 * np.pi * frequency) * 1e12
    ut1_bound = angle / OMEGA_EARTH
    recovered_phase = 2 * np.pi * frequency * baseline * angle / C_LIGHT
    assert np.isclose(recovered_phase, phase_limit)
    print(f'{name:15s} {angle_mas:14.3g} {delay_ps:11.3g} {ut1_bound:14.3g}')


三类示例的位置误差上限分别约为 34 mas、1.07 mas 和 0.0409 mas；延迟上限分别约为 19.8 ps、0.278 ps 和 3.31 ps。最不利 `UT1` 上限约为 $2.27\,\mathrm{ms}$、$71.4\,\mu\mathrm{s}$ 和 $2.72\,\mu\mathrm{s}$。它们不是设施规范，而是由所选基线、频率和 $10^\circ$ 门限直接得到的预算分量。

这个计算也解释了层级差异：常规成像软件必须正确读取时间与 EOP，不能用 UTC 数值代替 UT1；精密天体测量还必须证明剩余误差在目标-校准源差分中受到控制。fringe fitting 可以吸收部分站钟和几何残差，却不能自动恢复错误参考架、源结构或未记录的绝对位置基准。

![相位参考误差预算](figures/phase_reference_error_budget.png)

**图 3.5.3** 相位参考天体测量中的误差项随观测模式改变。连通阵列和 VLBI 都受热噪声、校准源位置、传播介质和模型误差限制，但 VLBI 对时钟、地球定向参数和源结构更敏感。图中数值是示意性的，重点是误差项的相对层级。

### 3.5.5 教学案例：相位参考位置为什么会随频率漂移

考虑一个高频相位参考观测。目标是一颗紧致活动星系核，校准源距离目标约一度。若低频图像和高频图像的峰值位置不完全重合，不能立即把差异解释为真实天体结构变化。首先要检查相位中心是否一致、频率设置是否共享同一参考频率、成像时是否使用相同的天球坐标定义、校准源位置是否来自同一参考架、以及两个频段的相位参考循环时间是否足够短。

随后需要区分三类效应。第一类是几何和时钟模型误差，它通常随基线、时间和频率呈现有规律的相位斜率。第二类是传播介质误差：电离层是色散介质，在低频更强；中性大气路径近似非色散，但同一路径误差产生的相位随频率增加，因此会限制高频相干时间。第三类是源结构误差，活动星系核的射电核心位置可能随频率改变，校准源本身也可能不是理想点源。若不做这些区分，把不同频率图像简单配准后测量谱指数或喷流结构，可能会把校准系统误差写入天体物理解释。

这个案例的教学价值在于，它把第 3 章的坐标和时间概念连接到第 8 章校准、第 9 章 VLBI 与高频实践。位置天文学不是成像前的附属准备，而是相位、延迟和科学位置测量的共同语言。

### 3.5.6 本节结论

时间标准、参考系、站坐标和相位中心共同决定干涉测量的几何模型。`UTC`、`UT1`、`TT/TAI`、`ICRS/ICRF`、`ITRS/ITRF` 和 EOP 分别承担不同角色。延迟误差通过 $|\Delta\phi|=2\pi\nu|\Delta\tau|$ 放大为相位误差，又通过 $|\Delta\theta|\simeq c|\Delta\tau|/B$ 进入天体测量。普通成像可以依赖经过验证的软件完成这些变换，但高频相位参考、VLBI 和精密位置测量必须显式记录参考架版本、参数历元、EOP 来源和相位跟踪约定。

***

下一节：[3.P 定量问题集](3_problem_set.ipynb)
